# 03. Эксперименты — модели, тюнинг, ансамбль, финальная модель

**План:**
1. Сплит + feature engineering (без лика, через `fit_transform_pipeline`).
2. Стратифицированный train-сэмпл 200k для скорости тюнинга и тяжёлых моделей.
3. Пять моделей «как есть»: `LinearRegression`, `Ridge`, `KNN`, `RandomForestRegressor`, `LightGBMRegressor`.
4. Подбор гиперпараметров для RF и LightGBM через `RandomizedSearchCV` (3-fold KFold).
5. Эксперимент с обучением на `log1p(target)` (с обратным преобразованием) — мотивация: тяжёлый правый хвост таргета.
6. Эксперимент с уменьшением размерности (PCA) на числовой подматрице.
7. Ансамбль `VotingRegressor(Ridge + RF + LGBM_tuned)`.
8. Финальный прогон победителя на **полном** train (без сэмплирования) и сохранение в `models/best_model.joblib`.
9. Сводная таблица для README и обоснование выбора финальной модели.

**Метрика для выбора финальной модели:** MAE на val (test трогаем только для финальной оценки победителя).


In [1]:
import sys, time, warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src import SEED
from src.preprocessing import (
    load_raw, clean, make_split, stratified_sample, fit_transform_pipeline, TARGET,
)
from src.modeling import (
    set_seed, evaluate, train_model, tune_lightgbm, tune_random_forest,
    build_ensemble, save_model, experiments_table,
)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
set_seed(SEED)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "bank_transactions.csv"
BEST_MODEL_PATH = PROJECT_ROOT / "models" / "best_model.joblib"


## 1. Загрузка, сплит, feature engineering

In [2]:
df = clean(load_raw(RAW_PATH))
train, val, test = make_split(df, test_size=0.2, val_size=0.1, seed=SEED)

train_sample = stratified_sample(train, n=200_000, seed=SEED)

(X_tr_full, y_tr_full), (X_val, y_val), (X_te, y_te), artifacts = fit_transform_pipeline(train, val, test)
(X_tr, y_tr), _, _, _ = fit_transform_pipeline(train_sample, val.head(1), test.head(1))
X_val = X_val.reindex(columns=X_tr.columns, fill_value=0)
X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

print("Sizes:", {"train_full": len(X_tr_full), "train_sample": len(X_tr), "val": len(X_val), "test": len(X_te)})
print("Features:", X_tr.shape[1])
print(list(X_tr.columns))


Sizes: {'train_full': 733411, 'train_sample': 200000, 'val': 104773, 'test': 209547}
Features: 16
['CustAccountBalance', 'Age', 'TransactionHour', 'TransactionDayOfWeek', 'TransactionMonth', 'LogBalance', 'LocationFreq', 'CustGender_F', 'CustGender_M', 'CustGender_unknown', 'AgeBucket_26-35', 'AgeBucket_36-45', 'AgeBucket_46-55', 'AgeBucket_56-65', 'AgeBucket_65+', 'AgeBucket_<=25']


**Защита от лика:**
- `engineer_features` в `fit_transform_pipeline` строит `LocationFreq` **только по train** и применяет тот же словарь к val/test.
- One-hot-категории из train сохраняются как опорные через `reindex` (в val/test недостающие колонки заполняются нулями).
- Никакие фичи не используют `TransactionAmount` — проверено в `tests/test.py::test_engineer_features_no_target_leakage`.
- `train_sample` берётся из `train`, val/test остаются нетронутыми.


## 2. Базовые модели «как есть» на сэмпле 200k

In [3]:
def time_fit_eval(name, hypothesis, params=None, X_tr_=X_tr, y_tr_=y_tr, X_val_=X_val, y_val_=y_val, notes=""):
    t0 = time.time()
    model = train_model(name, X_tr_, y_tr_, params=params)
    fit_seconds = time.time() - t0
    scores = evaluate(model, X_val_, y_val_)
    return {
        "model": name,
        "hypothesis": hypothesis,
        "params": str(params or {}),
        **scores,
        "fit_seconds": round(fit_seconds, 1),
        "notes": notes,
    }, model

results = []
fitted = {}

for name, hyp, params in [
    ("linear", "Линейная без регуляризации на полных фичах", None),
    ("ridge", "Линейная с L2-регуляризацией", {"alpha": 1.0}),
    ("knn", "Непараметрика, k=15", {"n_neighbors": 15}),
    ("random_forest", "Деревья из коробки", {"n_estimators": 200, "max_depth": 20}),
    ("lightgbm", "Бустинг по умолчанию", None),
]:
    row, model = time_fit_eval(name, hyp, params)
    results.append(row); fitted[name] = model
    print(f"{name:>14s}  MAE={row['MAE']:.1f}  RMSE={row['RMSE']:.1f}  R2={row['R2']:.3f}  fit={row['fit_seconds']}s")


        linear  MAE=1773.8  RMSE=6231.7  R2=0.018  fit=0.0s
         ridge  MAE=1773.8  RMSE=6231.7  R2=0.018  fit=0.1s


           knn  MAE=1882.8  RMSE=6359.9  R2=-0.023  fit=0.1s


 random_forest  MAE=1864.1  RMSE=6018.5  R2=0.084  fit=17.3s


      lightgbm  MAE=1786.9  RMSE=6180.6  R2=0.034  fit=1.3s


## 3. Тюнинг гиперпараметров (RandomizedSearchCV, 3-fold)

In [4]:
t0 = time.time()
rf_tuned, rf_best = tune_random_forest(X_tr, y_tr, n_iter=8, cv_folds=3)
print("RF best params:", rf_best, "time:", round(time.time()-t0,1), "s")


RF best params: {'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.8, 'max_depth': 10} time: 152.7 s


In [5]:
t0 = time.time()
lgbm_tuned, lgbm_best = tune_lightgbm(X_tr, y_tr, n_iter=15, cv_folds=3)
print("LGBM best params:", lgbm_best, "time:", round(time.time()-t0,1), "s")


LGBM best params: {'subsample': 0.85, 'reg_lambda': 0.1, 'reg_alpha': 0.1, 'num_leaves': 31, 'n_estimators': 800, 'min_child_samples': 50, 'learning_rate': 0.03, 'colsample_bytree': 0.7} time: 111.1 s


In [6]:
for name, model in [("random_forest_tuned", rf_tuned), ("lightgbm_tuned", lgbm_tuned)]:
    scores = evaluate(model, X_val, y_val)
    results.append({
        "model": name,
        "hypothesis": f"{name.split('_')[0].upper()} с подбором гиперпараметров (RandomizedSearchCV)",
        "params": str(rf_best if name.startswith('random') else lgbm_best),
        **scores,
        "fit_seconds": None,
        "notes": "tuned via RandomizedSearchCV",
    })
    fitted[name] = model
    print(f"{name:>22s}  MAE={scores['MAE']:.1f}  RMSE={scores['RMSE']:.1f}  R2={scores['R2']:.3f}")


   random_forest_tuned  MAE=1722.4  RMSE=6055.9  R2=0.073


        lightgbm_tuned  MAE=1758.6  RMSE=6170.2  R2=0.037


## 4. Эксперимент: обучение на `log1p(target)`

Мотивация: распределение таргета сильно скошено (skew ~47, max 1.56M, медиана 459).
Обучение на `log1p(y)` штрафует **относительную** ошибку, что естественно для финансовых сумм.
Метрики оцениваются на исходной шкале после `expm1` обратного преобразования.


In [7]:
y_tr_log = np.log1p(y_tr)
t0 = time.time()
lgbm_log = train_model("lightgbm", X_tr, y_tr_log, params=lgbm_best)
fit_seconds = time.time() - t0
preds = np.clip(np.expm1(lgbm_log.predict(X_val)), a_min=0, a_max=None)
from src.modeling import metrics as _metrics
scores = _metrics(y_val, preds)
results.append({
    "model": "lightgbm_log_target",
    "hypothesis": "Тот же LGBM с тюненными гиперами, но обучен на log1p(target)",
    "params": str(lgbm_best),
    **scores,
    "fit_seconds": round(fit_seconds, 1),
    "notes": "expm1 + clip(0)",
})
fitted["lightgbm_log_target"] = lgbm_log
print(f"lightgbm_log_target  MAE={scores['MAE']:.1f}  RMSE={scores['RMSE']:.1f}  R2={scores['R2']:.3f}")


lightgbm_log_target  MAE=1356.0  RMSE=6327.6  R2=-0.013


## 5. Уменьшение размерности (PCA)

После feature engineering фич немного (~12–16), но критерий CP1 требует «эксперимента с уменьшением размерности».
Применяем PCA к числовому подмножеству и обучаем на нём LightGBM. Сравним долю объяснённой дисперсии и MAE.


In [8]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr.astype(float))
X_val_scaled = scaler.transform(X_val.astype(float))

pca_full = PCA(random_state=SEED).fit(X_tr_scaled)
explained = pca_full.explained_variance_ratio_.cumsum()
n_for_95 = int(np.searchsorted(explained, 0.95) + 1)
print("Cumulative explained variance:", np.round(explained, 3))
print(f"Компонент для 95% дисперсии: {n_for_95} из {X_tr.shape[1]}")


Cumulative explained variance: [0.14  0.267 0.372 0.445 0.515 0.582 0.647 0.712 0.774 0.835 0.894 0.952
 0.997 1.    1.    1.   ]
Компонент для 95% дисперсии: 12 из 16


In [9]:
pca = PCA(n_components=n_for_95, random_state=SEED)
X_tr_pca = pca.fit_transform(X_tr_scaled)
X_val_pca = pca.transform(X_val_scaled)

t0 = time.time()
lgbm_pca = train_model("lightgbm", X_tr_pca, y_tr, params=None)
fit_seconds = time.time() - t0
scores = _metrics(y_val, lgbm_pca.predict(X_val_pca))
results.append({
    "model": "lightgbm_pca",
    "hypothesis": f"LGBM на PCA({n_for_95} компонент, 95% дисперсии)",
    "params": "default",
    **scores,
    "fit_seconds": round(fit_seconds, 1),
    "notes": f"{X_tr.shape[1]}->{n_for_95} components",
})
print(f"lightgbm_pca  MAE={scores['MAE']:.1f}  RMSE={scores['RMSE']:.1f}  R2={scores['R2']:.3f}")


lightgbm_pca  MAE=2947.1  RMSE=6851.7  R2=-0.187


## 6. Ансамбль — VotingRegressor

In [10]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler as SS
from sklearn.pipeline import Pipeline as PL
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

ridge_est = PL([("scaler", SS(with_mean=False)), ("model", Ridge(alpha=1.0, random_state=SEED))])
rf_est = RandomForestRegressor(**{**rf_best, "random_state": SEED, "n_jobs": -1})
lgbm_est = LGBMRegressor(**{**lgbm_best, "random_state": SEED, "n_jobs": -1, "verbose": -1})

ensemble = build_ensemble([("ridge", ridge_est), ("rf", rf_est), ("lgbm", lgbm_est)])
t0 = time.time()
ensemble.fit(X_tr, y_tr)
fit_seconds = time.time() - t0
scores = evaluate(ensemble, X_val, y_val)
results.append({
    "model": "voting_ensemble",
    "hypothesis": "Усреднение Ridge + RF_tuned + LGBM_tuned",
    "params": "equal weights",
    **scores,
    "fit_seconds": round(fit_seconds, 1),
    "notes": "VotingRegressor",
})
fitted["voting_ensemble"] = ensemble
print(f"voting_ensemble  MAE={scores['MAE']:.1f}  RMSE={scores['RMSE']:.1f}  R2={scores['R2']:.3f}")


voting_ensemble  MAE=1732.2  RMSE=6104.8  R2=0.057


## 7. Сводная таблица экспериментов

In [11]:
table = experiments_table(results).round(2)
table.sort_values("MAE", inplace=True)
print(table.to_string(index=False))


              model                                                   hypothesis                                                                                                                                                                   params     MAE    RMSE    R2    MAPE  fit_seconds                        notes
lightgbm_log_target Тот же LGBM с тюненными гиперами, но обучен на log1p(target) {'subsample': 0.85, 'reg_lambda': 0.1, 'reg_alpha': 0.1, 'num_leaves': 31, 'n_estimators': 800, 'min_child_samples': 50, 'learning_rate': 0.03, 'colsample_bytree': 0.7} 1355.98 6327.63 -0.01  540.98          2.5              expm1 + clip(0)
random_forest_tuned       RANDOM с подбором гиперпараметров (RandomizedSearchCV)                                                               {'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.8, 'max_depth': 10} 1722.39 6055.88  0.07 1886.07          NaN tuned via RandomizedSearchCV
    voting_ensemble               

## 8. Выбор финальной модели и переобучение на полных данных

Финальная модель — победитель по MAE на val. Переобучаем её на **полных** train-данных (без сэмплирования) и оцениваем на test.


In [12]:
best_row = table.iloc[0]
best_name = best_row["model"]
print("Победитель по MAE на val:", best_name, "(val MAE =", best_row["MAE"], ")")


Победитель по MAE на val: lightgbm_log_target (val MAE = 1355.98 )


In [13]:
# Переобучение победителя на полных данных
def rebuild_winner(name):
    if name == "lightgbm_tuned":
        return LGBMRegressor(**{**lgbm_best, "random_state": SEED, "n_jobs": -1, "verbose": -1})
    if name == "random_forest_tuned":
        return RandomForestRegressor(**{**rf_best, "random_state": SEED, "n_jobs": -1})
    if name == "lightgbm_log_target":
        return ("log_target", LGBMRegressor(**{**lgbm_best, "random_state": SEED, "n_jobs": -1, "verbose": -1}))
    if name == "voting_ensemble":
        return build_ensemble([
            ("ridge", PL([("scaler", SS(with_mean=False)), ("model", Ridge(alpha=1.0, random_state=SEED))])),
            ("rf", RandomForestRegressor(**{**rf_best, "random_state": SEED, "n_jobs": -1})),
            ("lgbm", LGBMRegressor(**{**lgbm_best, "random_state": SEED, "n_jobs": -1, "verbose": -1})),
        ])
    if name == "lightgbm":
        return LGBMRegressor(random_state=SEED, n_jobs=-1, verbose=-1)
    if name == "ridge":
        return PL([("scaler", SS(with_mean=False)), ("model", Ridge(alpha=1.0, random_state=SEED))])
    if name == "lightgbm_pca":
        raise RuntimeError("lightgbm_pca shouldn't be selected as final without saving the PCA transformer too.")
    raise ValueError(f"Don't know how to rebuild {name}")

reb = rebuild_winner(best_name)
log_target_mode = isinstance(reb, tuple) and reb[0] == "log_target"
final_model = reb[1] if log_target_mode else reb

t0 = time.time()
y_target = np.log1p(y_tr_full) if log_target_mode else y_tr_full
final_model.fit(X_tr_full, y_target)
print("Final fit on full train:", round(time.time()-t0, 1), "s")


Final fit on full train: 4.1 s


In [14]:
def predict_final(X):
    raw = final_model.predict(X)
    if log_target_mode:
        return np.clip(np.expm1(raw), 0, None)
    return raw

final_val = _metrics(y_val, predict_final(X_val))
final_test = _metrics(y_te, predict_final(X_te))
final_summary = pd.DataFrame({"val": final_val, "test": final_test}).round(2)
print("FINAL MODEL:", best_name)
print(final_summary)


FINAL MODEL: lightgbm_log_target
          val     test
MAE   1346.99  1343.92
RMSE  6294.90  6799.36
R2      -0.00    -0.00
MAPE   497.61   495.78


In [15]:
save_model({
    "name": best_name,
    "model": final_model,
    "log_target": log_target_mode,
    "feature_columns": list(X_tr_full.columns),
    "lgbm_params": lgbm_best,
    "rf_params": rf_best,
}, BEST_MODEL_PATH)
print("Saved:", BEST_MODEL_PATH)


Saved: C:\Users\egorm\PycharmProjects\hsemlcourse-classroom-0b8e58-hseml-group-project-project-template\models\best_model.joblib


## 9. Итоговая таблица для README

(скопировать в README в раздел «Результаты»)


In [16]:
baseline_val = {"MAE": 1823.26, "RMSE": 6274.13, "R2": 0.00}  # из 02_baseline.ipynb

readme_rows = [
    {"Модель": "Baseline LinearRegression (val)", **baseline_val},
    {"Модель": f"{best_name} (val)", **{k: round(v, 2) for k, v in final_val.items() if k in {'MAE','RMSE','R2'}}},
    {"Модель": f"{best_name} (test)", **{k: round(v, 2) for k, v in final_test.items() if k in {'MAE','RMSE','R2'}}},
]
readme_table = pd.DataFrame(readme_rows)
print(readme_table.to_string(index=False))


                         Модель     MAE    RMSE   R2
Baseline LinearRegression (val) 1823.26 6274.13  0.0
      lightgbm_log_target (val) 1346.99 6294.90 -0.0
     lightgbm_log_target (test) 1343.92 6799.36 -0.0


## 10. Обоснование выбора финальной модели

- **Метрика выбора:** MAE на val (см. `01_eda.ipynb` — обоснование диктуется тяжёлым правым хвостом таргета).
- Победитель: модель с минимальным MAE из таблицы выше; в типичной конфигурации это `lightgbm_tuned` или `voting_ensemble`.
- Обоснование, что не выбран бустинг на `log1p`-таргете: смотрим строку `lightgbm_log_target` в таблице — если её MAE сравним или хуже, оставляем модель в исходной шкале (проще интерпретация и деплой). Если лучше — берём её.
- Уменьшение размерности (PCA) приведено как требуемый эксперимент. На таких данных (≈15 фич, в основном категориальные one-hot и frequency) PCA как правило ухудшает MAE, потому что разрушает разреженную структуру; результат фиксируем для отчёта.
- Ансамбль (`voting_ensemble`) принимаем как финальную модель только если он улучшает MAE более чем на 1–2% относительно лучшей одиночной (правило из плана). В противном случае — лучшая одиночная (проще, дешевле, легче деплоить).
